In [ ]:
from __future__ import annotations

import asyncio
import json
import re
import unicodedata
import warnings
from dataclasses import dataclass
from datetime import date, datetime, timezone
from enum import Enum
from hashlib import sha256
from pathlib import Path
from time import perf_counter
from typing import Annotated, Any, Literal, TypeAlias, TypedDict
from uuid import uuid4

import pandas as pd
from pydantic import (
    BaseModel,
    ConfigDict,
    Field,
    StringConstraints,
    TypeAdapter,
    field_validator,
    model_validator,
)
from typing_extensions import Self

# Pydantic AI se importa de forma opcional para que el contrato y las pruebas
# deterministas puedan ejecutarse también en entornos sin acceso al modelo.
try:
    from pydantic_ai import Agent
    from pydantic_ai.models.ollama import OllamaModel
    from pydantic_ai.output import NativeOutput
    from pydantic_ai.providers.ollama import OllamaProvider
    from pydantic_ai.usage import RunUsage, UsageLimits
    PYDANTIC_AI_AVAILABLE = True
except ModuleNotFoundError:
    Agent = Any
    OllamaModel = Any
    NativeOutput = None
    OllamaProvider = Any
    PYDANTIC_AI_AVAILABLE = False

    @dataclass
    class RunUsage:  # fallback exclusivo para pruebas deterministas
        requests: int = 0
        input_tokens: int = 0
        output_tokens: int = 0
        total_tokens: int = 0

    class UsageLimits:
        def __init__(self, request_limit: int) -> None:
            self.request_limit = request_limit


def validate_required_columns(
    dataframe: pd.DataFrame,
    required_columns: set[str],
) -> None:
    missing = required_columns - set(dataframe.columns)
    if missing:
        raise ValueError(f"Faltan columnas obligatorias: {sorted(missing)}")


def find_project_root(start: Path | None = None) -> Path:
    """Localiza la raíz del proyecto buscando pyproject.toml."""

    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError(
        f"No se encontró pyproject.toml desde {start}. "
        "Ejecuta el notebook dentro del repositorio del TFM."
    )


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
SILVER_DIR = DATA_DIR / "silver"
SILVER_BOE_AI_DIR = SILVER_DIR / "boe_ai"

BOE_CANDIDATES_DOCS_TEXT_PATH = (
    SILVER_DIR
    / "boe_candidates_docs_text"
    / "boe_candidates_docs_text.parquet"
)

BOE_AI_EXTRACTIONS_PATH = SILVER_BOE_AI_DIR / "boe_ai_extractions.parquet"
BOE_AI_EXTRACTION_ATTEMPTS_PATH = (
    SILVER_BOE_AI_DIR / "boe_ai_extractions_attempts.parquet"
)

BOE_AI_REVIEW_QUEUE_PATH = SILVER_BOE_AI_DIR / "boe_ai_review_queue.parquet"
BOE_AI_MANUAL_REVIEWS_PATH = SILVER_BOE_AI_DIR / "boe_ai_manual_reviews.parquet"
BOE_AI_QUALITY_METRICS_PATH = SILVER_BOE_AI_DIR / "boe_ai_quality_metrics.parquet"
BOE_AI_MANUAL_REVIEW_DIR = PROJECT_ROOT / "config" / "manual_reviews" / "boe_ai"

# Tablas derivadas. Solo generation_asset_mentions alimentará la agrupación.
PUBLICATION_EVENTS_PATH = SILVER_BOE_AI_DIR / "publication_events.parquet"
GENERATION_ASSET_MENTIONS_PATH = (
    SILVER_BOE_AI_DIR / "generation_asset_mentions.parquet"
)
GENERATION_ASSET_NAMES_PATH = (
    SILVER_BOE_AI_DIR / "generation_asset_names.parquet"
)
ASSOCIATED_COMPONENTS_PATH = (
    SILVER_BOE_AI_DIR / "associated_components.parquet"
)
ASSOCIATED_COMPONENT_NAMES_PATH = (
    SILVER_BOE_AI_DIR / "associated_component_names.parquet"
)
ASSOCIATED_COMPONENT_GENERATION_LINKS_PATH = (
    SILVER_BOE_AI_DIR / "associated_component_generation_links.parquet"
)
ADMINISTRATIVE_ACTIONS_PATH = (
    SILVER_BOE_AI_DIR / "administrative_actions.parquet"
)
ADMINISTRATIVE_ACTION_TARGETS_PATH = (
    SILVER_BOE_AI_DIR / "administrative_action_targets.parquet"
)
PARTICIPANT_MENTIONS_PATH = (
    SILVER_BOE_AI_DIR / "participant_mentions.parquet"
)
LOCATION_MENTIONS_PATH = SILVER_BOE_AI_DIR / "location_mentions.parquet"
GENERATION_RELATIONS_PATH = (
    SILVER_BOE_AI_DIR / "generation_asset_relations.parquet"
)
TECHNICAL_MENTIONS_PATH = SILVER_BOE_AI_DIR / "technical_mentions.parquet"
CASE_FILE_REFERENCES_PATH = (
    SILVER_BOE_AI_DIR / "case_file_references.parquet"
)

## 11. Producción, revisión manual y regeneración de tablas (opt-in)

La producción no se bloquea por documentos excepcionales. Los intentos no
validados se escriben en `boe_ai_review_queue.parquet`. Para corregir uno:

1. ejecuta `create_manual_review_file(review_queue, "BOE-...")`;
2. edita el JSON creado en `config/manual_reviews/boe_ai/`;
3. cambia `review_status` a `manually_validated`, indica `reviewer`,
   `review_notes` y `reviewed_at_utc`, y corrige `corrected_extraction`;
4. ejecuta esta sección con `REFRESH_REVIEW_WORKFLOW=True`.

El JSON se valida con el mismo contrato y contra el BOE. Después, la revisión
manual se integra automáticamente con las extracciones automáticas válidas.


### Interpretación del piloto

`OK events=0` significa únicamente que la salida cumple el contrato como documento no relevante. Desde la versión 23 el piloto también compara esa decisión con `pilot_scope_labels_100.csv`. El piloto solo se considera superado si la exactitud de alcance y la validación estructural son ambas del 100 %.


Los descartes deterministas de alta precisión (contratación pública, infraestructura gasista y almacenamiento autónomo sin planta asociada) se guardan como extracciones válidas con `processing_stage=deterministic_scope_guard` y no pasan por la cola de revisión.


In [ ]:

RUN_PRODUCTION_EXTRACTION = False
RESET_PRODUCTION_OUTPUTS = False
RESET_QUALITY_METRICS = False
REFRESH_REVIEW_WORKFLOW = False
REGENERATE_FLAT_TABLES = False

PRODUCTION_BOE_IDS: tuple[str, ...] = ()
PRODUCTION_MAX_DOCUMENTS: int | None = None
PRODUCTION_MIN_AUTO_VALIDATION_RATE = 0.95


if RUN_PRODUCTION_EXTRACTION or REFRESH_REVIEW_WORKFLOW:
    all_candidates = load_and_prepare_candidates()

    if RESET_PRODUCTION_OUTPUTS:
        for path in (
            BOE_AI_EXTRACTION_ATTEMPTS_PATH,
            BOE_AI_EXTRACTIONS_PATH,
            BOE_AI_REVIEW_QUEUE_PATH,
        ):
            path.unlink(missing_ok=True)
    if RESET_QUALITY_METRICS:
        BOE_AI_QUALITY_METRICS_PATH.unlink(missing_ok=True)

    attempts = load_ai_extraction_attempts()
    manual_reviews = load_manual_review_files(
        all_candidates,
        attempts,
        review_dir=BOE_AI_MANUAL_REVIEW_DIR,
        output_path=BOE_AI_MANUAL_REVIEWS_PATH,
    )

    run_candidates = all_candidates.copy()
    if PRODUCTION_BOE_IDS:
        run_candidates = run_candidates.loc[
            run_candidates["identificador"].astype(str).isin(PRODUCTION_BOE_IDS)
        ].copy()
    if PRODUCTION_MAX_DOCUMENTS is not None:
        run_candidates = run_candidates.head(PRODUCTION_MAX_DOCUMENTS).copy()

    if RUN_PRODUCTION_EXTRACTION:
        _, pending = build_pending_candidates(
            run_candidates,
            attempts,
            manual_reviews=manual_reviews,
        )
        production_result = await run_and_finalize_extractions(
            pending,
            all_candidates,
            agent=agent,
            attempts_path=BOE_AI_EXTRACTION_ATTEMPTS_PATH,
            current_path=BOE_AI_EXTRACTIONS_PATH,
            review_queue_path=BOE_AI_REVIEW_QUEUE_PATH,
            quality_metrics_path=BOE_AI_QUALITY_METRICS_PATH,
            manual_reviews=manual_reviews,
            run_scope="production",
            minimum_auto_validation_rate=PRODUCTION_MIN_AUTO_VALIDATION_RATE,
            checkpoint_every=CHECKPOINT_EVERY,
        )
        attempts = production_result["all_attempts"]
        current_extractions = production_result["current_extractions"]
        review_queue = production_result["review_queue"]
        quality_metric = production_result["quality_metric"]
    else:
        current_extractions = select_best_valid_extractions(
            attempts=attempts,
            source_df=all_candidates,
            manual_reviews=manual_reviews,
        )
        save_parquet_atomic(current_extractions, BOE_AI_EXTRACTIONS_PATH)

        review_queue = build_review_queue(
            attempts=attempts,
            source_df=all_candidates,
            manual_reviews=manual_reviews,
        )
        save_parquet_atomic(review_queue, BOE_AI_REVIEW_QUEUE_PATH)

        quality_metric = build_quality_metric(
            attempts=attempts,
            source_df=all_candidates,
            manual_reviews=manual_reviews,
            run_scope="production",
            minimum_auto_validation_rate=PRODUCTION_MIN_AUTO_VALIDATION_RATE,
        )
        append_quality_metric(
            quality_metric,
            BOE_AI_QUALITY_METRICS_PATH,
        )

    display(current_extractions[
        [
            "identificador_boe",
            "selection_source",
            "document_scope",
            "n_publication_events",
            "n_generation_assets",
            "n_associated_components",
            "n_administrative_actions",
        ]
    ])
    display(review_queue)
    display(quality_metric)


if REGENERATE_FLAT_TABLES:
    current_extractions = normalise_ai_extraction_attempts_log(
        pd.read_parquet(BOE_AI_EXTRACTIONS_PATH)
    )
    flattened_tables = save_flattened_extractions(current_extractions)
    for table_name, dataframe in flattened_tables.items():
        print(f"{table_name}: {len(dataframe):,} filas")
